# Campisi 债券基金业绩归因模型

基于华泰金工研报《基于债券基金持仓的Campisi归因模型》复现。

## 模型原理

Campisi模型将债券基金收益率分解为三部分：

$$R = y \times dt + (-MD) \times dy_{treasury} + (-MD) \times dy_{credit}$$

其中：
- $y$：期初债券到期收益率
- $dt$：期初距上一次付息的时间间隔比例
- $MD$：期初修正久期
- $dy_{treasury}$：期间国债利率变化
- $dy_{credit}$：期间信用利差变化

In [ ]:
# 导入模块
import sys
import os
sys.path.insert(0, r'C:\Users\chenh\.qclaw\workspace\campisi_bond_attribution')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from config import setup_chinese_font, OUTPUT_DIR
from source.data_loader import (
    get_fund_bond_holdings,
    get_bond_info,
    get_treasury_yield_curve,
)
from source.yield_curve import YieldCurve
from source.campisi_model import CampisiAttribution
from source.plot import (
    plot_attribution_pie,
    plot_bond_contribution,
    plot_duration_distribution,
    plot_attribution_report,
)

# 设置中文字体
setup_chinese_font()

print('模块导入成功！')

## 1. 获取债券基金持仓数据

In [ ]:
# 选择分析基金
fund_code = '110017'  # 易方达增强回报A
fund_name = '易方达增强回报A'

# 获取持仓
holdings = get_fund_bond_holdings(fund_code)

print(f'持仓债券数: {len(holdings)}')
holdings.head(10)

## 2. 获取债券基本信息

In [ ]:
# 获取债券信息
bond_codes = holdings['bond_code'].tolist()
bond_info = get_bond_info(bond_codes)

print(f'债券信息数: {len(bond_info)}')
bond_info.head()

## 3. 获取国债收益率曲线

In [ ]:
from datetime import datetime, timedelta

# 分析区间
end_date = datetime.now().strftime('%Y%m%d')
start_date = (datetime.now() - timedelta(days=90)).strftime('%Y%m%d')

# 获取收益率曲线
curve_start = get_treasury_yield_curve(start_date, curve_type='国债')
curve_end = get_treasury_yield_curve(end_date, curve_type='国债')

print(f'期初收益率曲线:')
print(curve_start)

In [ ]:
# 构建YieldCurve对象
yc_start = YieldCurve(curve_start['term'].values, curve_start['yield_rate'].values)
yc_end = YieldCurve(curve_end['term'].values, curve_end['yield_rate'].values)

# 查看特定期限收益率
for term in [1, 2, 5, 10]:
    y_start = yc_start.get_yield(term)
    y_end = yc_end.get_yield(term)
    print(f'{term}年期: {y_start:.2f}% -> {y_end:.2f}% (变化: {y_end-y_start:.2f}bp)')

## 4. 执行Campisi归因分析

In [ ]:
# 执行归因分析
analyzer = CampisiAttribution()

results = analyzer.analyze(
    holdings=holdings,
    bond_info=bond_info,
    treasury_curve_start=yc_start,
    treasury_curve_end=yc_end,
    holding_period_days=90
)

# 获取摘要
summary = analyzer.get_summary()

print('\n归因分析完成！')

In [ ]:
# 显示归因摘要
print('=' * 50)
print(f'{fund_name} - Campisi归因分析结果')
print('=' * 50)
print(f'总收益:          {summary["total_return"]:.4f} ({summary["total_return"]*100:.2f}%)')
print(f'\n票息效应:        {summary["coupon_contrib"]:.4f} ({summary["coupon_pct"]:.1f}%)')
print(f'国债利率效应:    {summary["treasury_contrib"]:.4f} ({summary["treasury_pct"]:.1f}%)')
print(f'信用利差效应:    {summary["credit_contrib"]:.4f} ({summary["credit_pct"]:.1f}%)')
print(f'\n持仓债券数:      {summary["n_bonds"]}')
print(f'平均久期:        {summary["avg_duration"]:.2f}')
print(f'平均YTM:         {summary["avg_ytm"]*100:.2f}%')

In [ ]:
# 显示详细结果
results.head(10)

## 5. 可视化

In [ ]:
# 归因饼图
plot_attribution_pie(summary, title=f'{fund_name} - Campisi归因分解')

In [ ]:
# 票息效应贡献
plot_bond_contribution(results, effect='coupon', top_n=10)

In [ ]:
# 国债利率效应贡献
plot_bond_contribution(results, effect='treasury', top_n=10)

In [ ]:
# 信用利差效应贡献
plot_bond_contribution(results, effect='credit', top_n=10)

In [ ]:
# 久期分布
plot_duration_distribution(results)

In [ ]:
# 综合报告
plot_attribution_report(summary, results, fund_name=fund_name)

## 6. 保存结果

In [ ]:
# 保存详细结果
results_path = os.path.join(OUTPUT_DIR, 'campisi_detailed_results.csv')
results.to_csv(results_path, index=False, encoding='utf-8-sig')
print(f'详细结果已保存: {results_path}')

# 保存摘要
summary_path = os.path.join(OUTPUT_DIR, 'campisi_summary.csv')
pd.DataFrame([summary]).to_csv(summary_path, index=False, encoding='utf-8-sig')
print(f'归因摘要已保存: {summary_path}')